# 03 — Đánh giá kết quả suy luận

## Mục tiêu và dữ liệu
Chấm một lần chạy DeepSeek-R1-Distill theo đáp án chuẩn và trình bày độ chính xác, độ dài, ngôn ngữ và các loại lỗi. Chạy notebook riêng cho từng `run_id`. Mở notebook từ gốc bản sao dự án hoặc thư mục `notebooks/` và cài `requirements.txt` trong môi trường notebook.

Điền đường dẫn tới lần chạy và chính tệp dataset đã dùng khi sinh. Module kiểm tra SHA-256 với manifest. Nếu lần chạy chưa hoàn tất, kết quả tạm dùng những batch đã lưu và báo lượt còn thiếu. Không có dữ liệu giả mặc định. 

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

import runpy

cwd = Path.cwd()
helper = next((base / 'src/r1_project_root.py' for base in (cwd, *cwd.parents)
               if (base / 'src/r1_project_root.py').is_file()), None)
if helper is None:
    matches = list(cwd.glob('*/src/r1_project_root.py'))
    if len(matches) != 1:
        raise FileNotFoundError('Không xác định được repository. Hãy clone dự án hoặc chuyển cwd vào bản clone.')
    helper = matches[0]
project_root = runpy.run_path(str(helper))['find_project_root']()
sys.path.insert(0, str(project_root))
root = project_root
from src.evaluation import (
    accuracy_metrics, error_analysis, language_analysis, length_accuracy_relationship,
    load_evaluation_inputs, read_jsonl, reasoning_length_metrics, score_predictions,
)

RUN_DIR = None  # Điền đường dẫn thư mục chứa manifest.json
DATASET_PATH = None  # Điền đường dẫn CSV hoặc JSONL đã dùng ở notebook 02
REVIEW_PATH = None  # Tệp JSONL nhận xét do người chấm lập riêng cho lần chạy này
COUNT_REASONING_TOKENS = True  # Tải tokenizer tại revision ghi trong manifest

if not RUN_DIR or not DATASET_PATH:
    raise ValueError('Điền RUN_DIR và DATASET_PATH của dữ liệu thực nghiệm.')
manifest, questions, predictions = load_evaluation_inputs(root / RUN_DIR, root / DATASET_PATH)
reviews = read_jsonl(root / REVIEW_PATH) if REVIEW_PATH else []
count_tokens = None
if COUNT_REASONING_TOKENS:
    from transformers import AutoTokenizer

    spec = manifest['model']
    tokenizer = AutoTokenizer.from_pretrained(spec['id'], revision=spec['tokenizer_revision'])

    def count_tokens(text):
        return len(tokenizer.encode(text, add_special_tokens=False))

repeats = len(manifest['inference']['seeds'])
rows = score_predictions(predictions, questions, repeats, reviews=reviews, count_tokens=count_tokens)
print(f"Lần chạy {manifest['run_id']} | {manifest['model']['id']}")

## Accuracy và các trường hợp cần rà soát

Điểm chính là tỷ lệ đúng trong một lần sinh. Khi đủ lượt, mọi câu có cùng số lần sinh nên tỷ lệ đúng chung bằng trung bình tỷ lệ đúng theo câu. Đáp án chắc chắn vắng mặt được chấm sai. Trích xuất mơ hồ, đầu ra bị cắt hoặc biểu thức ngoài phạm vi được đưa vào bảng rà soát. Khi còn lượt thiếu hoặc chưa chấm, bảng hiển thị cận tỷ lệ đúng trên các lượt quan sát được, cho hai tình huống toàn bộ ca chờ đều sai hoặc đều đúng. Đây là cận theo trạng thái chấm, không phải khoảng tin cậy.

Bộ so sánh hỗ trợ số thập phân, phân số và biểu thức số hữu tỉ với bốn phép tính, ngoặc và số mũ nguyên giới hạn. Dấu phẩy giữa hai chữ số được đọc là dấu thập phân. Bộ chấm chỉ áp dụng dung sai hoặc đáp án tương đương đã ghi trong dữ liệu. Đơn vị, biểu thức ký hiệu và các trường hợp có nhiều cách hiểu cần được rà soát theo rubric.

In [ ]:
overall = accuracy_metrics(rows, len(questions) * repeats)
display(pd.DataFrame([overall]))

strata = []
for field in ('topic', 'difficulty'):
    for label in sorted({item[field] for item in questions.values()}):
        subset = [row for row in rows if row[field] == label]
        expected = sum(item[field] == label for item in questions.values()) * repeats
        strata.append({'group': field, 'label': label, **accuracy_metrics(subset, expected)})
display(pd.DataFrame(strata))

pending = [row for row in rows if row['needs_manual_review']]
columns = ['question_id', 'generation_index', 'question', 'gold_answer',
           'extracted_answer', 'extraction_status', 'stop_reason', 'generated_response']
display(pd.DataFrame(pending, columns=columns))

## Độ dài và mối liên hệ với độ chính xác

Tổng token lấy từ inference. Token suy luận được ước lượng bằng cách mã hóa lại phần có ranh giới với tokenizer đúng revision. Cột này để trống khi không thấy ranh giới hoặc tắt tokenizer. Bảng tương quan point-biserial dùng các lượt có điểm và độ dài xác định, trình bày cả khi giữ và khi loại đầu ra chạm trần token. Số liệu mô tả trong checkpoint này có thể bị ảnh hưởng bởi độ khó, nhóm bài, đầu ra bị cắt và việc một câu có nhiều lần sinh. Độ dài không phải thước đo chất lượng suy luận.

In [ ]:
display(pd.DataFrame(reasoning_length_metrics(rows)).T)
relations = []
for field in ('reasoning_token_count', 'generated_token_count'):
    for exclude in (False, True):
        relations.append({'length_field': field, 'exclude_truncated': exclude,
                          **length_accuracy_relationship(rows, length_field=field,
                                                         exclude_truncated=exclude)})
display(pd.DataFrame(relations))

scored = pd.DataFrame([row for row in rows if row['correct'] is not None])
if not scored.empty:
    for field, title in [('reasoning_token_count', 'Token suy luận ước lượng'),
                         ('generated_token_count', 'Tổng token đã sinh')]:
        if not scored[field].notna().any():
            continue
        fig, ax = plt.subplots(figsize=(8, 3))
        for correct, label in [(True, 'Đúng'), (False, 'Sai')]:
            values = scored.loc[scored['correct'].eq(correct), field].dropna()
            if not values.empty:
                ax.hist(values, alpha=0.55, label=label)
        ax.set(xlabel=title, ylabel='Số lượt')
        ax.legend()
        plt.show()

## Ngôn ngữ và lỗi

Bộ nhận diện dùng từ chỉ dấu và ký tự đặc trưng tiếng Việt. Nhãn trộn Việt–Anh yêu cầu ít nhất hai chỉ dấu riêng biệt ở mỗi ngôn ngữ trong cùng phần. Chỉ số trộn toàn đầu ra còn tính trường hợp phần suy luận và phần cuối dùng hai ngôn ngữ khác nhau. Các đoạn quá ngắn, chỉ có số hoặc thiếu tín hiệu thuộc nhóm `unknown`. Heuristic có thể bỏ sót đoạn trộn ngắn, nhầm tên riêng hoặc thuật ngữ mượn và chưa nhận diện riêng các ngôn ngữ khác. Rà soát thủ công một mẫu trước khi diễn giải tỷ lệ.

Để chấm tay, dùng tệp JSONL cho riêng lần chạy. Mỗi dòng gồm `question_id` cùng kiểu với predictions, `generation_index`, tùy chọn `correct` dạng boolean và `error_tags` dạng danh sách. Danh sách rỗng nghĩa là đã rà soát và không gắn nhãn. Các nhãn gồm `knowledge` (kiến thức sai), `calculation` (thực hiện phép tính sai), `misunderstood_question` (hiểu sai ý hoặc ràng buộc đề), `inconsistent_reasoning` (các bước mâu thuẫn), `unsupported_conclusion` (kết luận đúng hoặc sai nhưng thiếu căn cứ phù hợp) và `answer_not_extracted` (không trích xuất được đáp án). Có thể gắn nhiều nhãn cho một lượt. Các nhãn nội dung do người chấm xác định.

In [ ]:
languages = language_analysis(rows)
for field in ('reasoning_language', 'final_language'):
    print(field)
    display(pd.DataFrame(languages[field]))
print('Tỷ lệ trộn trên toàn bộ lượt:', languages['mixed_anywhere_share_all'])

errors = error_analysis(rows)
display(pd.DataFrame({
    'count_reviewed': errors['manual_counts'],
    'share_among_reviewed': errors['manual_share_reviewed'],
    'share_among_reviewed_wrong': errors['manual_share_reviewed_wrong'],
}).fillna({'count_reviewed': 0}))
print(f"Đã gắn nhãn {errors['reviewed_count']}/{len(rows)} lượt.")
print(f"Đã gắn nhãn {errors['reviewed_wrong_count']}/{errors['wrong_count']} lượt sai.")
print('Tỷ lệ gặp vấn đề khi tự động trích xuất:', errors['automatic_extraction_issue_share_all'])

## Kết quả và diễn giải

Ghi nhận độ phủ của các lượt sinh, số ca chờ rà soát và tỷ lệ hoàn thành việc gắn nhãn trước khi diễn giải. Khi so sánh các checkpoint, dùng cùng bộ câu hỏi, cùng quy tắc chấm và báo cáo độ bao phủ; thống kê theo lượt ở đây phục vụ mô tả và không coi các lần sinh của cùng câu là mẫu độc lập. Notebook chưa có kết quả khi chưa chọn một lần chạy thật.

## Lưu điểm để trực quan hóa
Chạy cell sau để lưu điểm từng lượt và thiết kế mẫu vào thư mục lần chạy. Nếu chấm bổ sung hoặc inference tiếp tục sinh, chọn tên tệp mới rồi trỏ notebook 04 tới tệp đó.

In [ ]:
from src.evaluation_artifacts import save_evaluation

EVALUATION_FILENAME = 'evaluation.json'
print(save_evaluation(root / RUN_DIR / EVALUATION_FILENAME, manifest, questions, predictions, rows))